# 🛡️ Laboratorio IDS con Suricata — Análisis Offline de PCAPs
## Ejecución en VirtualBox/Vagrant · Entorno local

---

**Objetivo de la práctica:**

Ejecutar este cuaderno te permitirá:
- Verificar e interpretar la configuración de Suricata (`suricata.yaml`)
- Actualizar reglas con `suricata-update` y justificar el impacto
- Analizar **offline** dos PCAPs con perfiles de amenaza distintos
- Comparar resultados entre escaneos web y tráfico WannaCry/EternalBlue
- Redactar un informe técnico basado en evidencia real

**Pasos a ejecutar (en orden):**
1. ✅ Verificación del entorno e instalación de Suricata
2. ✅ Revisión detallada de `/etc/suricata/suricata.yaml`
3. ✅ Actualización de reglas con `suricata-update`
4. ✅ Análisis offline — PCAP 1: *webserver scans and probes*
5. ✅ Análisis offline — PCAP 2: *WannaCry / EternalBlue*
6. ✅ Análisis comparativo y propuestas de configuración
7. ✅ Template de informe técnico

> ⚠️ **Importante:** Ejecuta las celdas **en orden**. Si tu entorno no permite captura en vivo, céntrate en el modo offline con PCAP (es el objetivo principal de la práctica).

---
## Sección 1: Verificación del Entorno e Instalación de Suricata

Esta sección instala Suricata (si no está ya disponible), detecta dinámicamente el entorno de ejecución y valida que todos los componentes necesarios están operativos.

In [ ]:
import subprocess
import os
import sys
import glob
import json
import shutil
from pathlib import Path

# Función auxiliar para leer archivos protegidos
def read_file_safe(filepath: Path) -> str:
    """Lee un archivo, manejando PermissionError con sudo."""
    try:
        return filepath.read_text()
    except PermissionError:
        result = subprocess.run(['sudo', 'cat', str(filepath)],
                              capture_output=True, text=True)
        if result.returncode == 0:
            return result.stdout
        else:
            return ""
    except FileNotFoundError:
        return ""

# Detectar entorno de ejecución
def detect_environment():
    """Detecta si estamos en Colab, Vagrant/VirtualBox o entorno genérico."""
    if 'google.colab' in sys.modules or os.path.exists('/content'):
        return 'colab', Path('/content')
    if os.path.exists('/home/vagrant'):
        return 'vagrant', Path('/home/vagrant')
    return 'generic', Path.home()

ENV_TYPE, WORK_DIR = detect_environment()
PCAP_DIR    = WORK_DIR / 'pcaps'
RESULTS_DIR = WORK_DIR / 'suricata_results'
LOG_DIR     = Path('/var/log/suricata')
RULES_DIR   = Path('/var/lib/suricata/rules')
CONFIG_FILE = Path('/etc/suricata/suricata.yaml')

PCAP_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Entorno detectado:", ENV_TYPE)
print("Directorio de trabajo:", WORK_DIR)
print("Directorio PCAPs:", PCAP_DIR)
print("Directorio resultados:", RESULTS_DIR)
print("Configuración Suricata:", CONFIG_FILE)

In [ ]:
result = subprocess.run(['suricata', '--build-info'], capture_output=True, text=True)
for line in result.stdout.splitlines():
    if 'Version' in line or 'version' in line:
        print(line.strip())

# Verificar permisos sudo
sudo_check = subprocess.run(['sudo', '-n', 'true'], capture_output=True)
sudo_ok = sudo_check.returncode == 0
print("sudo sin contraseña:", "DISPONIBLE" if sudo_ok else "NO DISPONIBLE")

# Verificar rutas clave (con manejo de permisos)
paths_to_check = {
    'suricata.yaml': CONFIG_FILE,
    'Directorio de logs': LOG_DIR,
    'Directorio de reglas': RULES_DIR,
}
print("\n--- Rutas del sistema ---")
for name, path in paths_to_check.items():
    try:
        exists = path.exists()
    except PermissionError:
        result = subprocess.run(['sudo', 'test', '-e', str(path)], capture_output=True)
        exists = result.returncode == 0
    
    status = "OK" if exists else "NO ENCONTRADO"
    print(f"{name:25s}: {path} [{status}]")

# Verificar interfaz de red
iface_result = subprocess.run(['ip', 'route'], capture_output=True, text=True)
default_iface = None
for line in iface_result.stdout.splitlines():
    if 'default' in line:
        parts = line.split()
        if 'dev' in parts:
            default_iface = parts[parts.index('dev') + 1]
            break
print(f"\nInterfaz de red principal: {default_iface or 'No detectada'}")

---
## Sección 2: Revisión de `/etc/suricata/suricata.yaml`

### ¿Por qué importa esta configuración?

`suricata.yaml` es el archivo central de configuración de Suricata. Determina:
- **Qué tráfico** analizar (`HOME_NET`, `EXTERNAL_NET`)
- **Dónde buscar** las reglas de detección (`default-rule-path`, `rule-files`)
- **Cómo registrar** los eventos detectados (`outputs`: `fast.log`, `eve.json`)
- **Qué protocolos** decodificar y analizar

### Parámetros clave a comprender:

| Parámetro | Función | Impacto en detección |
|-----------|---------|----------------------|
| `HOME_NET` | Define la red "protegida" interna | Reglas `$HOME_NET` solo disparan cuando el destino/origen pertenece a esta red |
| `EXTERNAL_NET` | Define el tráfico "externo" | Reglas `$EXTERNAL_NET` disparan para tráfico fuera de HOME_NET |
| `default-rule-path` | Directorio base de reglas | Todas las rutas relativas en `rule-files` son relativas a este directorio |
| `rule-files` | Lista de archivos de reglas | Solo se cargan los archivos listados aquí |
| `fast.log` | Log de alertas en texto plano | Útil para consulta rápida; formato: `timestamp  [**] [sid:rev] firma [**] src->dst` |
| `eve.json` | Log JSON enriquecido | Contiene todos los metadatos: IPs, puertos, protocolos, payload, timestamps |

In [ ]:
# ── Leer y mostrar secciones clave de suricata.yaml ──────────────────────────
def read_yaml_section(filepath, keywords, context_lines=2):
    """Extrae líneas que contienen las palabras clave dadas, con contexto."""
    content = read_file_safe(filepath)
    if not content:
        print(f"❌ No se puede leer: {filepath}")
        return {}
    
    lines = content.splitlines()
    results = {}
    for kw in keywords:
        matches = []
        for i, line in enumerate(lines):
            if kw in line:
                start = max(0, i - context_lines)
                end   = min(len(lines), i + context_lines + 1)
                block = lines[start:end]
                matches.append((i + 1, block))
        results[kw] = matches
    return results

keywords = ['HOME_NET', 'EXTERNAL_NET', 'default-rule-path', 'rule-files',
            'fast.log', 'eve.json', 'community-id']

sections = read_yaml_section(CONFIG_FILE, keywords)

print("=" * 70)
print("CONFIGURACIÓN ACTUAL DE suricata.yaml (secciones relevantes)")
print("=" * 70)

for kw, matches in sections.items():
    if matches:
        print(f"\n▶ {kw}:")
        for lineno, block in matches[:2]:  # máximo 2 ocurrencias por keyword
            print(f"  [línea {lineno}]")
            for l in block:
                print(f"    {l}")
    else:
        print(f"\n⚠️  '{kw}' no encontrado en suricata.yaml")

In [ ]:
# ── Extraer y explicar HOME_NET / EXTERNAL_NET ───────────────────────────────
content = read_file_safe(CONFIG_FILE)
if content:
    home_lines = [l.strip() for l in content.splitlines() if 'HOME_NET:' in l]
    ext_lines  = [l.strip() for l in content.splitlines() if 'EXTERNAL_NET:' in l]
    rules_path = [l.strip() for l in content.splitlines() if 'default-rule-path' in l]
    rule_files = [l.strip() for l in content.splitlines() if 'suricata.rules' in l or
                  (l.strip().startswith('-') and '.rules' in l)]

    print("📋 CONFIGURACIÓN DE RED:")
    print(f"   HOME_NET     : {home_lines[0] if home_lines else 'No encontrado'}")
    print(f"   EXTERNAL_NET : {ext_lines[0] if ext_lines else 'No encontrado'}")
    print(f"   Rules path   : {rules_path[0] if rules_path else 'No encontrado'}")

    print("\n📋 ARCHIVOS DE REGLAS:")
    for rf in rule_files[:5]:
        print(f"   {rf}")

    print("\n" + "─" * 60)
    print("💡 EXPLICACIÓN PEDAGÓGICA:")
    print("""
  • HOME_NET define qué redes son "internas" o "protegidas".
    - Las reglas con $HOME_NET como destino detectan ataques HACIA tu red.
    - Si un PCAP tiene IPs fuera del rango de HOME_NET, muchas reglas
      dirigidas a tu red NO dispararan → posibles falsos negativos.

  • EXTERNAL_NET suele ser 'any' o '!$HOME_NET' (cualquier IP exterior).
    - Reglas con $EXTERNAL_NET como origen detectan tráfico desde fuera.
    - Si HOME_NET es muy amplio (ej: 'any'), EXTERNAL_NET queda vacío
      → reglas de origen externo nunca disparan → falsos negativos.

  • Impacto práctico:
    - Para los PCAPs de práctica, verifica que las IPs del PCAP estén
      cubiertas por HOME_NET o EXTERNAL_NET según corresponda.
    - Configurar HOME_NET como 'any' puede generar más alertas pero
      también más falsos positivos.
    """)
else:
    print("⚠️  No se pudo leer suricata.yaml")

In [ ]:
# ── Modificar suricata.yaml para usar suricata.rules ─────────────────────────
import re
import shutil as _shutil

TARGET_RULES_PATH = '/var/lib/suricata/rules'
TARGET_RULES_FILE = 'suricata.rules'

def modify_suricata_yaml(config_path: Path) -> bool:
    """
    Ajusta suricata.yaml para que:
      1. default-rule-path apunte a /var/lib/suricata/rules
      2. rule-files contenga solo suricata.rules
    Retorna True si se realizó algún cambio.
    """
    content = read_file_safe(config_path)
    if not content:
        print(f"❌ No se puede leer {config_path}")
        return False

    original = content

    # Backup
    backup = config_path.with_suffix('.yaml.bak')
    result_backup = subprocess.run(['sudo', 'test', '-e', str(backup)], capture_output=True)
    if result_backup.returncode != 0:
        subprocess.run(['sudo', 'cp', str(config_path), str(backup)], check=True)
        print(f"💾 Backup guardado en: {backup}")

    # 1. Ajustar default-rule-path
    content = re.sub(
        r'(default-rule-path:\s*).*',
        f'\g<1>{TARGET_RULES_PATH}',
        content
    )

    # 2. Insertar/reemplazar bloque rule-files
    rule_files_block = f"""rule-files:
  - {TARGET_RULES_FILE}"""

    if re.search(r'^rule-files:', content, re.MULTILINE):
        content = re.sub(
            r'^rule-files:.*?(?=^[a-z])',
            rule_files_block + '\n\n',
            content,
            flags=re.MULTILINE | re.DOTALL
        )
    else:
        content += f"\n{rule_files_block}\n"

    if content != original:
        # Escribir con sudo
        import tempfile, subprocess as _sp
        with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as tmp:
            tmp.write(content)
            tmp_path = tmp.name
        _sp.run(['sudo', 'cp', tmp_path, str(config_path)], check=True)
        _sp.run(['sudo', 'chmod', '640', str(config_path)], check=True)
        os.unlink(tmp_path)
        print("✅ suricata.yaml modificado correctamente")
        return True
    else:
        print("ℹ️  suricata.yaml ya estaba configurado correctamente")
        return False

changed = modify_suricata_yaml(CONFIG_FILE)

# ── Validar cambios ────────────────────────────────────────────────────────────
content_after = read_file_safe(CONFIG_FILE)
if content_after:
    rule_path_ok = TARGET_RULES_PATH in content_after
    rule_file_ok = TARGET_RULES_FILE in content_after
    print(f"\n🔍 Validación post-modificación:")
    print(f"   {'✅' if rule_path_ok else '❌'} default-rule-path = {TARGET_RULES_PATH}")
    print(f"   {'✅' if rule_file_ok else '❌'} rule-files contiene {TARGET_RULES_FILE}")

    # Mostrar sección modificada
    lines = content_after.splitlines()
    for i, line in enumerate(lines):
        if 'default-rule-path' in line or 'rule-files' in line or TARGET_RULES_FILE in line:
            print(f"   [línea {i+1}] {line}")
else:
    print("⚠️  No se puede validar los cambios - no se puede leer el archivo")

---
## Sección 3: Actualización de Reglas con `suricata-update`

### ¿Por qué actualizar reglas?

Suricata utiliza **firmas** (reglas) para detectar amenazas. Las reglas predeterminadas de instalación son limitadas. `suricata-update` descarga y gestiona colecciones de reglas de fuentes públicas:

- **ET Open (Emerging Threats)**: Las reglas más completas, mantenidas por la comunidad
- **PT Research**: Positive Technologies Research
- **OISF**: Open Information Security Foundation

### Impacto de suricata-update:

| Sin actualizar | Con suricata-update (ET Open) |
|----------------|-------------------------------|
| ~100-500 reglas básicas | 40.000–50.000 reglas |
| Detección limitada | Cobertura amplia de amenazas |
| Sin reglas WannaCry/EternalBlue | Firmas específicas MS17-010, SMBv1 |
| Sin reglas para escaneos recientes | Reglas para Nmap, Masscan, etc. |

> **Nota:** Más reglas = más capacidad de detección, pero también mayor consumo de CPU/RAM.


In [ ]:
def count_rules(rules_file: Path) -> dict:
    """Cuenta reglas activas, comentadas y total en un archivo .rules."""
    try:
        exists = rules_file.exists()
    except PermissionError:
        result = subprocess.run(['sudo', 'test', '-e', str(rules_file)], capture_output=True)
        exists = result.returncode == 0

    if not exists:
        return {'active': 0, 'commented': 0, 'total': 0}

    content = read_file_safe(rules_file)
    if not content:
        return {'active': 0, 'commented': 0, 'total': 0}

    lines = content.splitlines()
    active    = sum(1 for l in lines if l.strip().startswith('alert'))
    commented = sum(1 for l in lines if l.strip().startswith('#alert'))
    return {'active': active, 'commented': commented, 'total': active + commented}

rules_file = RULES_DIR / 'suricata.rules'
before = count_rules(rules_file)

try:
    file_exists = rules_file.exists()
except PermissionError:
    result = subprocess.run(['sudo', 'test', '-e', str(rules_file)], capture_output=True)
    file_exists = result.returncode == 0

print("=" * 70)
print("ESTADO DE REGLAS ANTES DE ACTUALIZAR")
print("=" * 70)
print(f"\nReglas activas:   {before['active']:>10,}")
print(f"Reglas comentadas: {before['commented']:>9,}")
print(f"Total:            {before['total']:>10,}")
print(f"Archivo:          {rules_file}")
print(f"Existe:           {'Si' if file_exists else 'No'}")

In [ ]:
# ── Ejecutar suricata-update ───────────────────────────────────────────────────
print(" Actualizando fuentes de reglas...")
print("\\n$ sudo suricata-update update-sources")
result_sources = subprocess.run(
    ['sudo', 'suricata-update', 'update-sources'],
    capture_output=True, text=True
)
if result_sources.returncode == 0:
    print("Fuentes de reglas actualizadas")
else:
    print(f" update-sources: {result_sources.stderr[:200]}")

print("\\nHabilitando ET Open...")
print("\\n$ sudo suricata-update enable-source et/open")
result_enable = subprocess.run(
    ['sudo', 'suricata-update', 'enable-source', 'et/open'],
    capture_output=True, text=True
)
if result_enable.returncode == 0:
    print(" ET Open habilitado")
else:
    print(f" {result_enable.stdout[:200] or result_enable.stderr[:200]}")

print("\\nDescargando y compilando reglas (puede tardar 1-2 minutos)...")
print(f"\\n$ sudo suricata-update --no-merge --output {RULES_DIR}")
result_update = subprocess.run(
    ['sudo', 'suricata-update', '--no-merge', '--output', str(RULES_DIR)],
    capture_output=True, text=True
)

# Mostrar resumen del output
output_lines = (result_update.stdout + result_update.stderr).splitlines()
for line in output_lines:
    if any(kw in line.lower() for kw in ['loaded', 'rules', 'enabled', 'disabled',
                                          'skipped', 'sources', 'fetching', 'ok', 'error']):
        print(f"  {line.strip()}")

if result_update.returncode == 0:
    print("\\n suricata-update completado")
else:
    print(f"\\n suricata-update terminó con código {result_update.returncode}")

In [ ]:
after = count_rules(rules_file)

print("=" * 70)
print("COMPARATIVA DE REGLAS")
print("=" * 70)
print(f"\n{'Metrica':<22} {'Antes':>10} {'Despues':>10} {'Delta':>10}")
print("-" * 55)
for key in ['active', 'commented', 'total']:
    delta = after[key] - before[key]
    delta_str = f"+{delta:,}" if delta >= 0 else f"{delta:,}"
    print(f"  {key.capitalize():<20} {before[key]:>10,} {after[key]:>10,} {delta_str:>10}")

print()
if after['active'] > before['active']:
    print(f"Se agregaron {after['active'] - before['active']:,} reglas activas nuevas")
elif after['active'] == 0:
    print("ADVERTENCIA: No se encontraron reglas. Verificar conectividad.")
else:
    print(f"Numero de reglas activas: {after['active']:,}")

# Validar configuración con suricata -T
print("\nValidando configuracion de Suricata...")
test_result = subprocess.run(
    ['sudo', 'suricata', '-T', '-c', str(CONFIG_FILE), '-v'],
    capture_output=True, text=True
)

if test_result.returncode == 0:
    for line in (test_result.stdout + test_result.stderr).splitlines():
        if any(kw in line for kw in ['Loading', 'rules', 'Configuration', 'Done', 'Error', 'OK']):
            print(f"  {line.strip()}")
    print("\nConfiguracion valida: suricata.yaml es sintacticamente correcto")
else:
    print(f"ERROR en configuracion:\n{test_result.stderr[:500]}")

---
## Sección 4: Análisis Offline — PCAP 1: Webserver Scans and Probes

### Descripción del caso

Este PCAP contiene tráfico capturado de **escaneos y sondeos a servidores web**.
El tráfico típico incluye:
- Escaneos de puertos (TCP SYN scan, Nmap, Masscan)
- Peticiones HTTP de reconocimiento (dirbusting, nikto)
- Intentos de explotar vulnerabilidades web conocidas
- Escaneos de versiones de servicios

### ¿Qué esperar de Suricata?

| Tipo de alerta esperada | Firma típica | Protocolo |
|------------------------|--------------|-----------|
| Escaneo de puertos | ET SCAN Nmap... / ET SCAN Masscan | TCP |
| Reconocimiento HTTP | ET POLICY HTTP HEAD Request | HTTP |
| Exploits web | ET WEB_SPECIFIC_APPS ... | HTTP |
| Reconocimiento de servicios | ET SCAN ... version scan | TCP |

In [ ]:
# URLs para descarga de PCAPs (verificadas y funcionales en Google Colab)
PCAP1_URL      = 'https://www.malware-traffic-analysis.net/2024/11/24/2024-11-24-webserver-scans-and-probes.pcap.zip'
PCAP1_ZIP      = PCAP_DIR / '2024-11-24-webserver-scans-and-probes.pcap.zip'
PCAP1_PASSWORD = 'infected_20241124'
PCAP1_NAME     = '2024-11-24-webserver-scans-and-probes.pcap'

PCAP2_URL      = 'https://www.malware-traffic-analysis.net/2017/05/18/2017-05-18-WannaCry-ransomware-using-EnternalBlue-exploit.pcap.zip'
PCAP2_ZIP      = PCAP_DIR / '2017-05-18-WannaCry-ransomware-using-EnternalBlue-exploit.pcap.zip'
PCAP2_PASSWORD = 'infected_20170518'
PCAP2_NAME     = '2017-05-18-WannaCry-ransomware-using-EnternalBlue-exploit.pcap'

def download_pcap(url: str, dest_zip: Path, pcap_password: str,
                  expected_name: str = None) -> Path:
    """Descarga, descomprime y valida un PCAP. Retorna la ruta al .pcap."""
    
    print(f"Procesando: {expected_name or url}")
    print("-" * 70)
    
    # Buscar si ya existe el PCAP descomprimido
    existing = list(PCAP_DIR.glob('*.pcap'))
    for p in existing:
        if expected_name and expected_name in p.name:
            print(f"PCAP encontrado: {p.name} ({p.stat().st_size / (1024*1024):.1f} MB)")
            return p

    # Descargar ZIP
    if not dest_zip.exists():
        print(f"Descargando: {url}")
        dl_result = subprocess.run(
            ['wget', '-q', '--show-progress', '-O', str(dest_zip), url],
            capture_output=True, text=True, timeout=600
        )
        if dl_result.returncode != 0:
            print(f"ERROR: Descarga fallida (codigo {dl_result.returncode})")
            print(f"Intenta descargar manualmente: {url}")
            print(f"Contraseña: {pcap_password}")
            return None
        print(f"ZIP descargado: {dest_zip.name} ({dest_zip.stat().st_size / (1024*1024):.1f} MB)")
    else:
        print(f"ZIP ya existe: {dest_zip.name}")

    # Descomprimir con contraseña
    print(f"Descomprimiendo con contraseña...")
    unzip_result = subprocess.run(
        ['unzip', '-P', pcap_password, '-o', str(dest_zip), '-d', str(PCAP_DIR)],
        capture_output=True, text=True, timeout=120
    )
    
    if unzip_result.returncode != 0:
        print(f"ERROR en descompresion: {unzip_result.stderr[:200]}")
        return None

    # Buscar PCAP resultante
    pcaps = list(PCAP_DIR.glob('*.pcap'))
    if pcaps:
        if expected_name:
            for p in pcaps:
                if expected_name in p.name:
                    print(f"OK: {p.name} ({p.stat().st_size / (1024*1024):.1f} MB)")
                    return p
        
        pcap = max(pcaps, key=lambda p: p.stat().st_mtime)
        print(f"OK: {pcap.name} ({pcap.stat().st_size / (1024*1024):.1f} MB)")
        return pcap
    else:
        print(f"ERROR: No se encontro PCAP en {PCAP_DIR}")
        return None

# Descargar PCAP 1
print("\n" + "=" * 70)
print("DESCARGA DE PCAP 1: Webserver Scans and Probes")
print("=" * 70)
pcap1_path = download_pcap(PCAP1_URL, PCAP1_ZIP, PCAP1_PASSWORD, PCAP1_NAME)
PCAP1_PATH = pcap1_path

In [ ]:
PCAP1_RESULTS = RESULTS_DIR / 'pcap1_webserver'
PCAP1_RESULTS.mkdir(parents=True, exist_ok=True)

def run_suricata_offline(pcap_path: Path, output_dir: Path,
                          config: Path = CONFIG_FILE) -> bool:
    """Ejecuta Suricata en modo offline sobre un PCAP dado."""
    # Safe file existence check
    try:
        pcap_exists = pcap_path.exists() if pcap_path else False
    except PermissionError:
        result = subprocess.run(['sudo', 'test', '-e', str(pcap_path)], capture_output=True)
        pcap_exists = result.returncode == 0

    if pcap_path is None or not pcap_exists:
        print(f"ERROR: PCAP no disponible: {pcap_path}")
        return False

    # Limpiar resultados previos
    for f in output_dir.glob('*.log'):
        f.unlink()
    for f in output_dir.glob('*.json'):
        f.unlink()

    print(f"Analizando: {pcap_path.name}")
    print(f"Resultados en: {output_dir}")
    print("Procesando PCAP (puede tardar)...")

    result = subprocess.run(
        ['sudo', 'suricata',
         '-r', str(pcap_path),
         '-c', str(config),
         '-l', str(output_dir),
         '-k', 'none',
         '--set', 'outputs.1.fast.enabled=yes'],
        capture_output=True, text=True, timeout=600
    )

    if result.returncode == 0 or output_dir.joinpath('fast.log').exists():
        fast_log = output_dir / 'fast.log'
        eve_json = output_dir / 'eve.json'
        
        # Safe file reading
        alerts = 0
        if fast_log.exists():
            fast_content = read_file_safe(fast_log)
            alerts = fast_content.count('[**]') if fast_content else 0
        
        events = 0
        if eve_json.exists():
            try:
                with open(eve_json, errors='replace') as f:
                    events = sum(1 for _ in f)
            except PermissionError:
                result_eve = subprocess.run(['sudo', 'wc', '-l', str(eve_json)],
                                           capture_output=True, text=True)
                if result_eve.returncode == 0:
                    events = int(result_eve.stdout.split()[0])

        print(f"\nAnalisis completado:")
        print(f"  fast.log: {'OK' if fast_log.exists() else 'NO'}")
        print(f"  eve.json: {'OK' if eve_json.exists() else 'NO'}")
        print(f"  Alertas: {alerts}")
        print(f"  Eventos: {events}")
        return True
    else:
        print(f"ERROR: Suricata fallo (codigo {result.returncode})")
        return False

if PCAP1_PATH:
    success_pcap1 = run_suricata_offline(PCAP1_PATH, PCAP1_RESULTS)
else:
    print("ERROR: PCAP 1 no disponible para analisis")
    success_pcap1 = False

In [ ]:
import re

def parse_fast_log(fast_log_path: Path, max_alerts: int = None) -> list:
    """Parsea fast.log y retorna lista de dicts con las alertas."""
    alerts = []
    
    # Safe file existence check
    try:
        exists = fast_log_path.exists()
    except PermissionError:
        result = subprocess.run(['sudo', 'test', '-e', str(fast_log_path)], capture_output=True)
        exists = result.returncode == 0
    
    if not exists:
        return alerts

    # Safe file reading
    content = read_file_safe(fast_log_path)
    if not content:
        return alerts

    pattern = re.compile(
        r'(\d{2}/\d{2}/\d{4}-[\d:]+\.\d+)\s+'   # timestamp
        r'\[\*\*\]\s+\[(.+?)\]\s+'              # [gid:sid:rev]
        r'(.+?)\s+\[\*\*\]\s+'                   # signature
        r'\[Classification:\s*(.+?)\]\s+'          # classification
        r'\[Priority:\s*(\d+)\]\s+'               # priority
        r'\{(\w+)\}\s+'                            # protocol
        r'([\d.]+:\d+)\s*->\s*([\d.]+:\d+)'     # src -> dst
    )

    for line in content.splitlines():
        m = pattern.match(line.strip())
        if m:
            alerts.append({
                'timestamp':      m.group(1),
                'sid':            m.group(2),
                'signature':      m.group(3),
                'classification': m.group(4),
                'priority':       int(m.group(5)),
                'protocol':       m.group(6),
                'src':            m.group(7),
                'dst':            m.group(8),
                'raw':            line.strip()
            })
        elif '[**]' in line:
            alerts.append({'raw': line.strip(), 'signature': line.strip(), 'priority': 99})

    if max_alerts is None:
        return alerts
    return alerts[:max_alerts]

fast1 = PCAP1_RESULTS / 'fast.log'
alerts1 = parse_fast_log(fast1)

print("=" * 70)
print("ALERTAS FAST.LOG - PCAP 1: WEBSERVER SCANS")
print("=" * 70)
print(f"\nTotal de alertas: {len(alerts1)}\n")

# Top firmas
from collections import Counter
sig_counter1 = Counter()

for a in alerts1:
    sig = a.get('signature', 'N/A')[:60]
    sig_counter1[sig] += 1

print("Principales firmas detectadas:")
print("-" * 70)
for sig, cnt in sig_counter1.most_common(15):
    print(f"{cnt:>5} ocurrencias | {sig}")

# Mostrar TODAS las alertas
print(f"\nDetalle de alertas ({len(alerts1)} total):")
print("-" * 70)
for i, a in enumerate(alerts1, 1):
    raw = a.get('raw', 'N/A')
    if len(raw) > 150:
        raw = raw[:147] + "..."
    print(f"{i:>4}. {raw}")

# Resumen de prioridades
prio_counter = Counter(a.get('priority', 99) for a in alerts1)
print("\nDistribucion por prioridad:")
print("-" * 70)
for p in sorted(prio_counter.keys()):
    label = {1: 'CRITICA', 2: 'ALTA', 3: 'MEDIA'}.get(p, f'NIVEL {p}')
    print(f"Prioridad {p} ({label:10s}): {prio_counter[p]:>6} alertas")

# Resumen por protocolo
proto_counter = Counter(a.get('protocol', 'N/A') for a in alerts1 if 'protocol' in a)
print("\nAlertas por protocolo:")
print("-" * 70)
for proto, cnt in proto_counter.most_common():
    print(f"{proto:>10s}: {cnt:>6} alertas")

In [ ]:
import pandas as pd

def load_eve_json(eve_path: Path, max_lines: int = None) -> pd.DataFrame:
    """Carga eve.json en un DataFrame, expandiendo el campo 'alert'."""
    records = []
    
    # Safe file existence check
    try:
        exists = eve_path.exists()
    except PermissionError:
        result = subprocess.run(['sudo', 'test', '-e', str(eve_path)], capture_output=True)
        exists = result.returncode == 0
    
    if not exists:
        return pd.DataFrame()

    # Try to open file safely
    try:
        with open(eve_path, errors='replace') as f:
            for i, line in enumerate(f):
                if max_lines and i >= max_lines:
                    break
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    except PermissionError:
        # Fall back to sudo
        result = subprocess.run(['sudo', 'cat', str(eve_path)],
                              capture_output=True, text=True)
        if result.returncode == 0:
            for i, line in enumerate(result.stdout.splitlines()):
                if max_lines and i >= max_lines:
                    break
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    pass

    if not records:
        return pd.DataFrame()

    df = pd.json_normalize(records, sep='_')
    return df

eve1_path = PCAP1_RESULTS / 'eve.json'
df1 = load_eve_json(eve1_path)

if not df1.empty:
    print("=" * 70)
    print("EVENTOS EVE.JSON - PCAP 1: WEBSERVER SCANS")
    print("=" * 70)
    print(f"\nTotal de eventos: {len(df1)}\n")

    # Tipos de eventos
    if 'event_type' in df1.columns:
        print("Distribucion por tipo de evento:")
        print("-" * 70)
        for etype, cnt in df1['event_type'].value_counts().items():
            print(f"{etype:>20s}: {cnt:>8} eventos")

    # Mostrar TODAS las alertas de forma ordenada
    if 'event_type' in df1.columns:
        df_alerts1 = df1[df1['event_type'] == 'alert'].copy()
        
        if not df_alerts1.empty:
            print(f"\nDETALLE DE ALERTAS ({len(df_alerts1)} total):")
            print("-" * 70)
            
            # Seleccionar columnas relevantes
            display_cols = ['timestamp', 'src_ip', 'src_port', 'dest_ip', 'dest_port', 'proto']
            alert_cols = ['alert_signature', 'alert_category', 'alert_severity']
            
            available_display = [c for c in display_cols if c in df_alerts1.columns]
            available_alert = [c for c in alert_cols if c in df_alerts1.columns]
            
            # Mostrar todas las alertas numeradas
            for i, (idx, row) in enumerate(df_alerts1.iterrows(), 1):
                timestamp = row.get('timestamp', 'N/A')
                src_ip = row.get('src_ip', 'N/A')
                src_port = row.get('src_port', 'N/A')
                dest_ip = row.get('dest_ip', 'N/A')
                dest_port = row.get('dest_port', 'N/A')
                proto = row.get('proto', 'N/A')
                signature = row.get('alert_signature', 'N/A')
                category = row.get('alert_category', 'N/A')
                severity = row.get('alert_severity', 'N/A')
                
                print(f"{i:>4}. [{timestamp}] {src_ip}:{src_port} -> {dest_ip}:{dest_port} ({proto})")
                print(f"      Firma: {signature}")
                print(f"      Categoria: {category} | Severidad: {severity}\n")

            # Resumen de protocolos
            print("Alertas por protocolo:")
            print("-" * 70)
            if 'proto' in df_alerts1.columns:
                for proto, cnt in df_alerts1['proto'].value_counts().items():
                    print(f"{proto:>20s}: {cnt:>8} alertas")

            # Top IPs origen
            if 'src_ip' in df_alerts1.columns:
                print("\nTop 10 IPs origen de alertas:")
                print("-" * 70)
                for rank, (ip, cnt) in enumerate(df_alerts1['src_ip'].value_counts().head(10).items(), 1):
                    print(f"{rank:>2}. {ip:>20s}: {cnt:>8} alertas")

            # Top firmas
            if 'alert_signature' in df_alerts1.columns:
                print("\nTop 10 firmas mas frecuentes:")
                print("-" * 70)
                for rank, (sig, cnt) in enumerate(df_alerts1['alert_signature'].value_counts().head(10).items(), 1):
                    sig_short = sig[:60] if len(str(sig)) > 60 else sig
                    print(f"{rank:>2}. {sig_short:60s}: {cnt:>8} alertas")
        else:
            print("\nNo se encontraron alertas")
    else:
        print("\nNo se encontro columna 'event_type'")
else:
    print("No se pudieron cargar eventos de eve.json")

In [ ]:
# ── Visualización de resultados PCAP 1 ────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

# Use inline backend for static output — compatible with all Jupyter environments (no ipympl required)
%matplotlib inline
plt.rcParams.update({'figure.figsize': (14, 5), 'figure.dpi': 100,
                     'font.size': 10})

if not df1.empty and 'event_type' in df1.columns:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle('PCAP 1: Webserver Scans and Probes — Análisis Suricata', fontsize=13)

    # ── Panel 1: Tipos de eventos ─────────────────────────────────────────────
    event_counts = df1['event_type'].value_counts().head(8)
    axes[0].barh(event_counts.index, event_counts.values, color='steelblue')
    axes[0].set_title('Tipos de Eventos')
    axes[0].set_xlabel('Cantidad')
    axes[0].invert_yaxis()

    # ── Panel 2: Top firmas ───────────────────────────────────────────────────
    df_al1 = df1[df1['event_type'] == 'alert'] if 'event_type' in df1.columns else pd.DataFrame()
    if not df_al1.empty and 'alert_signature' in df_al1.columns:
        top_sigs = df_al1['alert_signature'].value_counts().head(8)
        labels = [s[:35] + '...' if len(str(s)) > 35 else str(s) for s in top_sigs.index]
        axes[1].barh(labels, top_sigs.values, color='coral')
        axes[1].set_title('Top Firmas de Alerta')
        axes[1].set_xlabel('Cantidad')
        axes[1].invert_yaxis()
    else:
        axes[1].text(0.5, 0.5, 'Sin datos\nde alertas', ha='center', va='center',
                     transform=axes[1].transAxes)
        axes[1].set_title('Top Firmas de Alerta')

    # ── Panel 3: Protocolos ───────────────────────────────────────────────────
    if 'proto' in df1.columns:
        proto_counts = df1['proto'].value_counts().head(6)
        axes[2].pie(proto_counts.values,
                    labels=[str(p) for p in proto_counts.index],
                    autopct='%1.1f%%', startangle=90,
                    colors=plt.cm.Set3.colors[:len(proto_counts)])
        axes[2].set_title('Distribución de Protocolos')
    else:
        axes[2].text(0.5, 0.5, 'Sin datos\nde protocolos', ha='center', va='center',
                     transform=axes[2].transAxes)

    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'pcap1_analysis.png'), bbox_inches='tight')
    plt.show()
    print("📊 Gráfico guardado en:", RESULTS_DIR / 'pcap1_analysis.png')
else:
    print("⚠️  No hay datos suficientes para visualizar. Verifica el análisis del PCAP.")

---
## Sección 5: Análisis Offline — PCAP 2: WannaCry / EternalBlue

### Descripción del caso

Este PCAP histórico (mayo 2017) contiene tráfico capturado durante el ataque **WannaCry ransomware** que explotó la vulnerabilidad **EternalBlue (MS17-010)**.

### Contexto técnico

| Aspecto | Detalle |
|---------|---------|
| Vulnerabilidad | MS17-010 (SMBv1, puerto 445) |
| Exploit | EternalBlue (filtrado de la NSA) |
| Payload | WannaCry ransomware |
| Propagación | Auto-propagación por redes Windows |
| CVE | CVE-2017-0144, CVE-2017-0145 |

### ¿Qué esperar de Suricata?

| Tipo de alerta esperada | Firma ET Open típica | Protocolo |
|------------------------|---------------------|-----------|
| Explotación SMBv1 | `ET EXPLOIT MS17-010 EternalBlue` | SMB/TCP |
| Escaneo de SMB | `ET SCAN SMB Brute Force` | TCP/445 |
| Propagación de gusano | `ET WORM` | TCP |
| Tráfico C&C | `ET MALWARE` | TCP |
| Actividad DoublePulsar | `ET EXPLOIT DoublePulsar` | SMB |

> **Nota pedagógica:** La diferencia clave con el PCAP 1 es el **protocolo dominante** (SMB vs HTTP) y la **naturaleza de las alertas** (explotación activa vs reconocimiento).

In [ ]:
print("\n" + "=" * 70)
print("DESCARGA DE PCAP 2: WannaCry / EternalBlue")
print("=" * 70)

PCAP2_PATH = None

# Buscar si ya existe algún PCAP de WannaCry
for existing in PCAP_DIR.glob('*.pcap'):
    if 'WannaCry' in existing.name or 'EternalBlue' in existing.name or '2017' in existing.name:
        print(f"PCAP encontrado: {existing.name} ({existing.stat().st_size / (1024*1024):.1f} MB)")
        PCAP2_PATH = existing
        break

# Si no existe, descargarlo
if PCAP2_PATH is None:
    pcap2_path = download_pcap(PCAP2_URL, PCAP2_ZIP, PCAP2_PASSWORD, PCAP2_NAME)
    PCAP2_PATH = pcap2_path

if PCAP2_PATH:
    print(f"\nPCAP 2 disponible: {PCAP2_PATH.name}")
else:
    print("\nERROR: PCAP 2 no disponible")
    print(f"URL: {PCAP2_URL}")
    print(f"Contraseña: {PCAP2_PASSWORD}")
    print("Intenta descargar manualmente e coloca en:", PCAP_DIR)

In [ ]:
PCAP2_RESULTS = RESULTS_DIR / 'pcap2_wannacry'
PCAP2_RESULTS.mkdir(parents=True, exist_ok=True)

print("\n" + "=" * 70)
print("ANALISIS DE PCAP 2: WANNACRY / ETERNALBLUE")
print("=" * 70)

if PCAP2_PATH:
    success_pcap2 = run_suricata_offline(PCAP2_PATH, PCAP2_RESULTS)
else:
    print("ERROR: PCAP 2 no disponible para analisis")
    success_pcap2 = False

In [ ]:
fast2 = PCAP2_RESULTS / 'fast.log'
alerts2 = parse_fast_log(fast2)

print("=" * 70)
print("ALERTAS FAST.LOG - PCAP 2: WANNACRY / ETERNALBLUE")
print("=" * 70)
print(f"\nTotal de alertas: {len(alerts2)}\n")

sig_counter2 = Counter()
for a in alerts2:
    sig = a.get('signature', 'N/A')[:60]
    sig_counter2[sig] += 1

print("Principales firmas detectadas:")
print("-" * 70)
for sig, cnt in sig_counter2.most_common(15):
    print(f"{cnt:>5} ocurrencias | {sig}")

# Mostrar TODAS las alertas
print(f"\nDetalle de alertas ({len(alerts2)} total):")
print("-" * 70)
for i, a in enumerate(alerts2, 1):
    raw = a.get('raw', 'N/A')
    if len(raw) > 150:
        raw = raw[:147] + "..."
    print(f"{i:>4}. {raw}")

prio_counter2 = Counter(a.get('priority', 99) for a in alerts2)
print("\nDistribucion por prioridad:")
print("-" * 70)
for p in sorted(prio_counter2.keys()):
    label = {1: 'CRITICA', 2: 'ALTA', 3: 'MEDIA'}.get(p, f'NIVEL {p}')
    print(f"Prioridad {p} ({label:10s}): {prio_counter2[p]:>6} alertas")

# Resumen por protocolo
proto_counter2 = Counter(a.get('protocol', 'N/A') for a in alerts2 if 'protocol' in a)
print("\nAlertas por protocolo:")
print("-" * 70)
for proto, cnt in proto_counter2.most_common():
    print(f"{proto:>10s}: {cnt:>6} alertas")

In [ ]:
eve2_path = PCAP2_RESULTS / 'eve.json'
df2 = load_eve_json(eve2_path)

if not df2.empty:
    print("=" * 70)
    print("EVENTOS EVE.JSON - PCAP 2: WANNACRY / ETERNALBLUE")
    print("=" * 70)
    print(f"\nTotal de eventos: {len(df2)}\n")

    # Tipos de eventos
    if 'event_type' in df2.columns:
        print("Distribucion por tipo de evento:")
        print("-" * 70)
        for etype, cnt in df2['event_type'].value_counts().items():
            print(f"{etype:>20s}: {cnt:>8} eventos")

    # Mostrar TODAS las alertas de forma ordenada
    if 'event_type' in df2.columns:
        df_alerts2 = df2[df2['event_type'] == 'alert'].copy()
        
        if not df_alerts2.empty:
            print(f"\nDETALLE DE ALERTAS ({len(df_alerts2)} total):")
            print("-" * 70)
            
            # Mostrar todas las alertas numeradas
            for i, (idx, row) in enumerate(df_alerts2.iterrows(), 1):
                timestamp = row.get('timestamp', 'N/A')
                src_ip = row.get('src_ip', 'N/A')
                src_port = row.get('src_port', 'N/A')
                dest_ip = row.get('dest_ip', 'N/A')
                dest_port = row.get('dest_port', 'N/A')
                proto = row.get('proto', 'N/A')
                signature = row.get('alert_signature', 'N/A')
                category = row.get('alert_category', 'N/A')
                severity = row.get('alert_severity', 'N/A')
                
                print(f"{i:>4}. [{timestamp}] {src_ip}:{src_port} -> {dest_ip}:{dest_port} ({proto})")
                print(f"      Firma: {signature}")
                print(f"      Categoria: {category} | Severidad: {severity}\n")

            # Resumen de protocolos
            print("Alertas por protocolo:")
            print("-" * 70)
            if 'proto' in df_alerts2.columns:
                for proto, cnt in df_alerts2['proto'].value_counts().items():
                    print(f"{proto:>20s}: {cnt:>8} alertas")

            # Top IPs origen
            if 'src_ip' in df_alerts2.columns:
                print("\nTop 10 IPs origen de alertas:")
                print("-" * 70)
                for rank, (ip, cnt) in enumerate(df_alerts2['src_ip'].value_counts().head(10).items(), 1):
                    print(f"{rank:>2}. {ip:>20s}: {cnt:>8} alertas")

            # Top firmas
            if 'alert_signature' in df_alerts2.columns:
                print("\nTop 10 firmas mas frecuentes:")
                print("-" * 70)
                for rank, (sig, cnt) in enumerate(df_alerts2['alert_signature'].value_counts().head(10).items(), 1):
                    sig_short = sig[:60] if len(str(sig)) > 60 else sig
                    print(f"{rank:>2}. {sig_short:60s}: {cnt:>8} alertas")

            # Buscar firmas especificas de WannaCry
            print("\nFirmas relacionadas con WannaCry/EternalBlue:")
            print("-" * 70)
            wannacry_keywords = ['MS17-010', 'EternalBlue', 'DoublePulsar', 'SMB', 'EXPLOIT']
            for kw in wannacry_keywords:
                mask = df_alerts2['alert_signature'].str.contains(kw, case=False, na=False)
                count = mask.sum()
                if count > 0:
                    print(f"'{kw}': {count} alertas encontradas")
        else:
            print("\nNo se encontraron alertas")
    else:
        print("\nNo se encontro columna 'event_type'")
else:
    print("No se pudieron cargar eventos de eve.json")

In [ ]:
# ── Visualización de resultados PCAP 2 ────────────────────────────────────────
if not df2.empty and 'event_type' in df2.columns:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle('PCAP 2: WannaCry/EternalBlue — Análisis Suricata', fontsize=13)

    event_counts2 = df2['event_type'].value_counts().head(8)
    axes[0].barh(event_counts2.index, event_counts2.values, color='darkred')
    axes[0].set_title('Tipos de Eventos')
    axes[0].set_xlabel('Cantidad')
    axes[0].invert_yaxis()

    df_al2 = df2[df2['event_type'] == 'alert'] if 'event_type' in df2.columns else pd.DataFrame()
    if not df_al2.empty and 'alert_signature' in df_al2.columns:
        top_sigs2 = df_al2['alert_signature'].value_counts().head(8)
        labels2 = [s[:35] + '...' if len(str(s)) > 35 else str(s) for s in top_sigs2.index]
        axes[1].barh(labels2, top_sigs2.values, color='darkorange')
        axes[1].set_title('Top Firmas de Alerta')
        axes[1].set_xlabel('Cantidad')
        axes[1].invert_yaxis()
    else:
        axes[1].text(0.5, 0.5, 'Sin datos\nde alertas', ha='center', va='center',
                     transform=axes[1].transAxes)
        axes[1].set_title('Top Firmas de Alerta')

    if 'proto' in df2.columns:
        proto_counts2 = df2['proto'].value_counts().head(6)
        axes[2].pie(proto_counts2.values,
                    labels=[str(p) for p in proto_counts2.index],
                    autopct='%1.1f%%', startangle=90,
                    colors=plt.cm.Set2.colors[:len(proto_counts2)])
        axes[2].set_title('Distribución de Protocolos')
    else:
        axes[2].text(0.5, 0.5, 'Sin datos\nde protocolos', ha='center', va='center',
                     transform=axes[2].transAxes)

    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'pcap2_analysis.png'), bbox_inches='tight')
    plt.show()
    print("📊 Gráfico guardado en:", RESULTS_DIR / 'pcap2_analysis.png')
else:
    print("⚠️  No hay datos suficientes para visualizar el PCAP 2.")

---
## Sección 6: Análisis Comparativo — Webserver Scans vs WannaCry/EternalBlue

### ¿Por qué comparar estos dos PCAPs?

Estos dos casos representan **perfiles de amenaza opuestos**:

| Dimensión | PCAP 1: Webserver Scans | PCAP 2: WannaCry |
|-----------|------------------------|-----------------|
| **Fase del ataque** | Reconocimiento | Explotación activa |
| **Protocolo dominante** | HTTP/TCP | SMB (TCP/445) |
| **Objetivo** | Servidores web públicos | Sistemas Windows en red interna |
| **Técnica MITRE** | T1046 (Network Service Scanning) | T1210 (Exploitation of Remote Services) |
| **Severidad** | Media–Alta | Crítica |
| **Propagación** | No se propaga | Auto-propagante (gusano) |

Esta comparación te permitirá responder la pregunta del enunciado:
> *¿Qué diferencias observas entre el "perfil de actividad" detectado por Suricata en el PCAP de webserver scans and probes y el PCAP histórico de WannaCry/EternalBlue?*

In [ ]:
# ── Tabla comparativa de los dos PCAPs ────────────────────────────────────────
def get_pcap_stats(df: pd.DataFrame, pcap_name: str) -> dict:
    """Extrae estadísticas resumen de un DataFrame de eve.json."""
    if df.empty:
        return {'name': pcap_name, 'total_events': 0, 'alerts': 0,
                'top_protocol': 'N/A', 'top_signature': 'N/A',
                'unique_src_ips': 0, 'unique_dst_ips': 0,
                'categories': []}

    stats = {'name': pcap_name}
    stats['total_events'] = len(df)

    if 'event_type' in df.columns:
        df_al = df[df['event_type'] == 'alert']
        stats['alerts'] = len(df_al)

        if 'proto' in df.columns:
            stats['top_protocol'] = df['proto'].value_counts().index[0]                 if not df['proto'].empty else 'N/A'
        else:
            stats['top_protocol'] = 'N/A'

        if 'alert_signature' in df_al.columns and not df_al.empty:
            stats['top_signature'] = df_al['alert_signature'].value_counts().index[0]                 if not df_al['alert_signature'].empty else 'N/A'
        else:
            stats['top_signature'] = 'N/A'

        stats['unique_src_ips'] = df['src_ip'].nunique() if 'src_ip' in df.columns else 0
        stats['unique_dst_ips'] = df['dest_ip'].nunique() if 'dest_ip' in df.columns else 0

        if 'alert_category' in df_al.columns and not df_al.empty:
            stats['categories'] = df_al['alert_category'].value_counts().head(3).index.tolist()
        else:
            stats['categories'] = []
    else:
        stats.update({'alerts': 0, 'top_protocol': 'N/A', 'top_signature': 'N/A',
                      'unique_src_ips': 0, 'unique_dst_ips': 0, 'categories': []})

    return stats

stats1 = get_pcap_stats(df1, 'PCAP 1: Webserver Scans')
stats2 = get_pcap_stats(df2, 'PCAP 2: WannaCry/EternalBlue')

print("=" * 75)
print("TABLA COMPARATIVA — Webserver Scans vs WannaCry/EternalBlue")
print("=" * 75)
print(f"{'Métrica':<35} {'PCAP 1 (Scans)':<20} {'PCAP 2 (WannaCry)':<20}")
print("-" * 75)

comparisons = [
    ('Total eventos (eve.json)',  'total_events'),
    ('Total alertas',             'alerts'),
    ('Protocolo dominante',       'top_protocol'),
    ('IPs origen únicas',         'unique_src_ips'),
    ('IPs destino únicas',        'unique_dst_ips'),
]
for label, key in comparisons:
    v1 = str(stats1.get(key, 'N/A'))
    v2 = str(stats2.get(key, 'N/A'))
    print(f"  {label:<33} {v1:<20} {v2:<20}")

print(f"  {'Firma más frecuente':<33} {str(stats1['top_signature'])[:20]:<20} {str(stats2['top_signature'])[:20]:<20}")
print("=" * 75)

# Categorías de alerta
print("\n📂 Categorías de alerta detectadas:")
print(f"  PCAP 1: {', '.join(stats1['categories'][:3]) or 'N/A'}")
print(f"  PCAP 2: {', '.join(stats2['categories'][:3]) or 'N/A'}")

In [ ]:
# ── Visualización comparativa ─────────────────────────────────────────────────
has_data1 = not df1.empty
has_data2 = not df2.empty

if has_data1 or has_data2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Comparativa: Webserver Scans vs WannaCry/EternalBlue', fontsize=13)

    # ── Panel 1: Tipos de eventos side-by-side ───────────────────────────────
    all_types = set()
    if has_data1 and 'event_type' in df1.columns:
        all_types.update(df1['event_type'].value_counts().head(6).index.tolist())
    if has_data2 and 'event_type' in df2.columns:
        all_types.update(df2['event_type'].value_counts().head(6).index.tolist())

    all_types = sorted(all_types)
    x = range(len(all_types))
    w = 0.35

    counts1 = [df1['event_type'].value_counts().get(t, 0) if has_data1 and 'event_type' in df1.columns else 0
               for t in all_types]
    counts2 = [df2['event_type'].value_counts().get(t, 0) if has_data2 and 'event_type' in df2.columns else 0
               for t in all_types]

    bars1 = axes[0].bar([i - w/2 for i in x], counts1, w, label='PCAP 1 (Scans)', color='steelblue')
    bars2 = axes[0].bar([i + w/2 for i in x], counts2, w, label='PCAP 2 (WannaCry)', color='darkred')
    axes[0].set_xticks(list(x))
    axes[0].set_xticklabels(all_types, rotation=30, ha='right')
    axes[0].set_title('Tipos de Eventos por PCAP')
    axes[0].set_ylabel('Cantidad')
    axes[0].legend()

    # ── Panel 2: Top firmas únicas por PCAP ──────────────────────────────────
    top_pcap1_sigs = {}
    top_pcap2_sigs = {}

    if has_data1 and 'event_type' in df1.columns:
        df_al1 = df1[df1['event_type'] == 'alert']
        if 'alert_signature' in df_al1.columns and not df_al1.empty:
            top_pcap1_sigs = df_al1['alert_signature'].value_counts().head(5).to_dict()

    if has_data2 and 'event_type' in df2.columns:
        df_al2 = df2[df2['event_type'] == 'alert']
        if 'alert_signature' in df_al2.columns and not df_al2.empty:
            top_pcap2_sigs = df_al2['alert_signature'].value_counts().head(5).to_dict()

    if top_pcap1_sigs or top_pcap2_sigs:
        categories_labels = ['PCAP 1\n(Scans)', 'PCAP 2\n(WannaCry)']
        total_alerts = [stats1['alerts'], stats2['alerts']]
        colors_bar = ['steelblue', 'darkred']
        axes[1].bar(categories_labels, total_alerts, color=colors_bar, width=0.4)
        axes[1].set_title('Total de Alertas por PCAP')
        axes[1].set_ylabel('Número de alertas')
        for i, v in enumerate(total_alerts):
            axes[1].text(i, v + max(total_alerts) * 0.02, str(v),
                         ha='center', fontweight='bold')
    else:
        axes[1].text(0.5, 0.5, 'Sin datos\npara comparar', ha='center', va='center',
                     transform=axes[1].transAxes)

    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'comparison.png'), bbox_inches='tight')
    plt.show()
    print("📊 Gráfico comparativo guardado en:", RESULTS_DIR / 'comparison.png')
else:
    print("⚠️  No hay datos de ningún PCAP para comparar visualmente.")

In [ ]:
# ── Análisis de HOME_NET/EXTERNAL_NET e impacto en detección ─────────────────
print("=" * 70)
print("ANÁLISIS DE CONFIGURACIÓN — HOME_NET / EXTERNAL_NET")
print("=" * 70)

# Extraer valores actuales
home_net_value   = 'No configurado'
external_net_val = 'No configurado'

content = read_file_safe(CONFIG_FILE)
if content:
    for line in content.splitlines():
        if 'HOME_NET:' in line and not line.strip().startswith('#'):
            home_net_value = line.split(':', 1)[1].strip().strip('"')
        if 'EXTERNAL_NET:' in line and not line.strip().startswith('#'):
            external_net_val = line.split(':', 1)[1].strip().strip('"')

print(f"\n📋 Configuración actual:")
print(f"   HOME_NET     = {home_net_value}")
print(f"   EXTERNAL_NET = {external_net_val}")

# Detectar IPs en los PCAPs
def get_unique_ips(df: pd.DataFrame) -> set:
    ips = set()
    for col in ['src_ip', 'dest_ip']:
        if col in df.columns:
            ips.update(df[col].dropna().unique())
    return ips

ips1 = get_unique_ips(df1)
ips2 = get_unique_ips(df2)

if ips1 or ips2:
    print(f"\n🌐 IPs únicas en PCAP 1 (primeras 10): {list(ips1)[:10]}")
    print(f"🌐 IPs únicas en PCAP 2 (primeras 10): {list(ips2)[:10]}")

print("""
\n💡 IMPACTO DE HOME_NET EN LA DETECCIÓN:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. FALSOS NEGATIVOS por HOME_NET mal configurado:
   Si las IPs del PCAP no están en HOME_NET, reglas como:
     alert tcp $EXTERNAL_NET any -> $HOME_NET 445 (msg:"ET EXPLOIT MS17-010")
   NO dispararán, aunque el tráfico sea malicioso.

   → Solución: Configurar HOME_NET = 'any' para análisis forense de PCAPs
     externos, o incluir específicamente los rangos del PCAP.

2. FALSOS POSITIVOS con HOME_NET = any:
   Si HOME_NET = any, EXTERNAL_NET = !$HOME_NET = vacío → sin tráfico externo.
   Muchas reglas de detección de ataques externos dejarán de disparar.
   
   → Solución óptima: Usar el rango real de IPs del PCAP como HOME_NET.

3. PROPUESTA DE AJUSTE PARA ESTOS PCAPs:
   Para análisis offline forense:
   
   # Para cubrir todos los rangos privados + públicos comunes:
   HOME_NET: "[192.168.0.0/16,10.0.0.0/8,172.16.0.0/12,any]"
   EXTERNAL_NET: "any"
   
   Esto maximiza la detección en ambos PCAPs sin crear conflictos.

4. AJUSTES DE REGLAS PROPUESTOS:
   - Para reducir falsos positivos en PCAP 1 (scans web):
     * Deshabilitar reglas de "Policy" de baja severidad
     * Aumentar threshold en reglas de escaneo: 'threshold: type both, track by_src, count 50, seconds 60'
   
   - Para PCAP 2 (WannaCry):
     * Habilitar reglas específicas: et/open category 'emerging-exploit'
     * Activar alertas para SMBv1: 'proto:smb' en suricata.yaml → smb: enabled: yes
""")

---
## Sección 7: Template de Informe Técnico

### Instrucciones para el alumno

Basándote en las evidencias generadas por las secciones anteriores, completa el informe técnico a continuación. **Copia los fragmentos reales** de `fast.log` y `eve.json` que encontraste al ejecutar el notebook.

---

### ✍️ Pregunta del enunciado

> **¿Qué diferencias observas entre el "perfil de actividad" detectado por Suricata en el PCAP de webserver scans and probes y el PCAP histórico de WannaCry/EternalBlue, y qué ajustes (de configuración y/o de reglas) propondrías para reducir falsos positivos sin perder capacidad de detección?**

---

### 1) Evidencia — Alertas y Eventos JSON

**Instrucciones:** Copia al menos 2 alertas de `fast.log` y 2 eventos de `eve.json` por cada PCAP.

#### PCAP 1: Webserver Scans and Probes

**Alerta fast.log #1:**
```
[PEGA AQUÍ UNA ALERTA DE fast.log DEL PCAP 1]
Ejemplo: 11/24/2024-10:23:45.123456  [**] [1:2010935:4] ET SCAN Nmap Scripting Engine [**] [Classification: Web Application Attack] [Priority: 1] {TCP} 192.168.1.100:52341 -> 10.0.0.5:80
```

**Alerta fast.log #2:**
```
[PEGA AQUÍ OTRA ALERTA DE fast.log DEL PCAP 1]
```

**Evento eve.json #1:**
```json
[PEGA AQUÍ UN EVENTO JSON DEL PCAP 1]
Campos importantes: timestamp, event_type, src_ip, dest_ip, proto, alert.signature, alert.category
```

**Evento eve.json #2:**
```json
[PEGA AQUÍ OTRO EVENTO JSON DEL PCAP 1]
```

#### PCAP 2: WannaCry / EternalBlue

**Alerta fast.log #1:**
```
[PEGA AQUÍ UNA ALERTA DE fast.log DEL PCAP 2]
Ejemplo: 05/18/2017-03:04:05.678901  [**] [1:2023047:5] ET EXPLOIT MS17-010 EternalBlue Exploit [**] [Classification: Attempted Administrator Privilege Gain] [Priority: 1] {TCP} 192.168.1.50:12345 -> 192.168.1.200:445
```

**Alerta fast.log #2:**
```
[PEGA AQUÍ OTRA ALERTA DE fast.log DEL PCAP 2]
```

**Evento eve.json #1:**
```json
[PEGA AQUÍ UN EVENTO JSON DEL PCAP 2]
```

**Evento eve.json #2:**
```json
[PEGA AQUÍ OTRO EVENTO JSON DEL PCAP 2]
```

---

### 2) Interpretación — Comparativa de Perfiles

**Instrucciones:** Redacta 3–5 párrafos comparando los dos PCAPs. Usa la tabla comparativa de la Sección 6.

**Preguntas guía:**
- ¿Qué protocolo domina en cada PCAP? ¿Por qué es relevante?
- ¿Qué tipo de actividad predomina: reconocimiento, explotación, propagación?
- ¿Cómo deduces esto de las firmas de `fast.log` y los `event_type` de `eve.json`?
- ¿Cuál de los dos representa mayor riesgo inmediato y por qué?

> **[ESCRIBE TU INTERPRETACIÓN AQUÍ]**

---

### 3) Configuración — suricata.yaml y variables de red

**Instrucciones:** Relaciona tu análisis con la configuración de Suricata.

**Preguntas guía:**
- ¿Cómo afectan `HOME_NET` y `EXTERNAL_NET` a las alertas que observaste?
- ¿Hubo alertas que esperabas ver y no aparecieron? ¿A qué puede deberse?
- ¿Qué cambio hiciste en `rule-files`? ¿Por qué es necesario?
- ¿Cuántas reglas cargó `suricata-update`? ¿Sin él, habrías detectado WannaCry?
- ¿Qué ajuste de `HOME_NET` propones para reducir falsos positivos en PCAP 1?

> **[ESCRIBE TU ANÁLISIS DE CONFIGURACIÓN AQUÍ]**

---

### 4) Propuestas de Optimización

**Propuestas concretas basadas en tu análisis:**

| # | Problema observado | Ajuste propuesto | Impacto esperado |
|---|-------------------|-----------------|-----------------|
| 1 | [ej: Demasiadas alertas de baja prioridad en PCAP 1] | [ej: Deshabilitar categoría "Potentially Bad Traffic"] | [ej: -30% alertas, misma cobertura crítica] |
| 2 | [ej: MS17-010 no detectado sin suricata-update] | [ej: Automatizar suricata-update diario con cron] | [ej: Cobertura actualizada contra amenazas recientes] |
| 3 | [COMPLETA] | [COMPLETA] | [COMPLETA] |

In [ ]:
print("=" * 70)
print("RESUMEN FINAL DE LA PRACTICA")
print("=" * 70)

# Safe file existence checks
try:
    suricata_ok = bool(shutil.which('suricata'))
except:
    suricata_ok = False

try:
    config_exists = CONFIG_FILE.exists()
except PermissionError:
    result = subprocess.run(['sudo', 'test', '-e', str(CONFIG_FILE)], capture_output=True)
    config_exists = result.returncode == 0

config_content = read_file_safe(CONFIG_FILE)
rules_yaml_ok = 'suricata.rules' in config_content if config_content else False

try:
    rules_file_exists = (RULES_DIR / 'suricata.rules').exists()
except PermissionError:
    result = subprocess.run(['sudo', 'test', '-e', str(RULES_DIR / 'suricata.rules')], capture_output=True)
    rules_file_exists = result.returncode == 0

try:
    pcap1_results_ok = (PCAP1_RESULTS / 'fast.log').exists()
except PermissionError:
    result = subprocess.run(['sudo', 'test', '-e', str(PCAP1_RESULTS / 'fast.log')], capture_output=True)
    pcap1_results_ok = result.returncode == 0

try:
    pcap2_results_ok = (PCAP2_RESULTS / 'fast.log').exists()
except PermissionError:
    result = subprocess.run(['sudo', 'test', '-e', str(PCAP2_RESULTS / 'fast.log')], capture_output=True)
    pcap2_results_ok = result.returncode == 0

summary = {
    'Suricata instalado':              suricata_ok,
    'suricata.yaml configurado':       config_exists,
    'Rutas de reglas ajustadas':       rules_yaml_ok,
    'suricata-update ejecutado':       rules_file_exists,
    'PCAP 1 analizado':                pcap1_results_ok,
    'PCAP 2 analizado':                pcap2_results_ok,
    'Analisis comparativo completado': True,
}

rules_count = count_rules(RULES_DIR / 'suricata.rules')

print("\nEstatus de componentes:")
print("-" * 70)
for task, done in summary.items():
    status = "OK" if done else "FALLO"
    print(f"{task:40s}: {status}")

print(f"\nReglas cargadas: {rules_count['active']:,} activas")

alerts_1 = len(parse_fast_log(PCAP1_RESULTS / 'fast.log')) if pcap1_results_ok else 0
alerts_2 = len(parse_fast_log(PCAP2_RESULTS / 'fast.log')) if pcap2_results_ok else 0

print("\nAlertas generadas:")
print("-" * 70)
print(f"PCAP 1 (Webserver Scans):    {alerts_1:>8} alertas")
print(f"PCAP 2 (WannaCry/EternalBlue): {alerts_2:>8} alertas")
print(f"TOTAL:                        {alerts_1 + alerts_2:>8} alertas")

print(f"\nResultados guardados en: {RESULTS_DIR}")
print("\nPractica completada. Ver seccion 7 para informe tecnico.")

---
## Sección 7: Template de Informe Técnico

### Instrucciones para el alumno

Basándote en las evidencias generadas por las secciones anteriores, completa el informe técnico a continuación. **Copia los fragmentos reales** de `fast.log` y `eve.json` que encontraste al ejecutar el notebook.

---

### ✍️ Pregunta del enunciado

> **¿Qué diferencias observas entre el "perfil de actividad" detectado por Suricata en el PCAP de webserver scans and probes y el PCAP histórico de WannaCry/EternalBlue, y qué ajustes (de configuración y/o de reglas) propondrías para reducir falsos positivos sin perder capacidad de detección?**

---

### 1) Evidencia — Alertas y Eventos JSON

**Instrucciones:** Copia al menos 2 alertas de `fast.log` y 2 eventos de `eve.json` por cada PCAP.

#### PCAP 1: Webserver Scans and Probes

**Alerta fast.log #1:**
```
[PEGA AQUÍ UNA ALERTA DE fast.log DEL PCAP 1]
Ejemplo: 11/24/2024-10:23:45.123456  [**] [1:2010935:4] ET SCAN Nmap Scripting Engine [**] [Classification: Web Application Attack] [Priority: 1] {TCP} 192.168.1.100:52341 -> 10.0.0.5:80
```

**Alerta fast.log #2:**
```
[PEGA AQUÍ OTRA ALERTA DE fast.log DEL PCAP 1]
```

**Evento eve.json #1:**
```json
[PEGA AQUÍ UN EVENTO JSON DEL PCAP 1]
Campos importantes: timestamp, event_type, src_ip, dest_ip, proto, alert.signature, alert.category
```

**Evento eve.json #2:**
```json
[PEGA AQUÍ OTRO EVENTO JSON DEL PCAP 1]
```

#### PCAP 2: WannaCry / EternalBlue

**Alerta fast.log #1:**
```
[PEGA AQUÍ UNA ALERTA DE fast.log DEL PCAP 2]
Ejemplo: 05/18/2017-03:04:05.678901  [**] [1:2023047:5] ET EXPLOIT MS17-010 EternalBlue Exploit [**] [Classification: Attempted Administrator Privilege Gain] [Priority: 1] {TCP} 192.168.1.50:12345 -> 192.168.1.200:445
```

**Alerta fast.log #2:**
```
[PEGA AQUÍ OTRA ALERTA DE fast.log DEL PCAP 2]
```

**Evento eve.json #1:**
```json
[PEGA AQUÍ UN EVENTO JSON DEL PCAP 2]
```

**Evento eve.json #2:**
```json
[PEGA AQUÍ OTRO EVENTO JSON DEL PCAP 2]
```

---

### 2) Interpretación — Comparativa de Perfiles

**Instrucciones:** Redacta 3–5 párrafos comparando los dos PCAPs. Usa la tabla comparativa de la Sección 6.

**Preguntas guía:**
- ¿Qué protocolo domina en cada PCAP? ¿Por qué es relevante?
- ¿Qué tipo de actividad predomina: reconocimiento, explotación, propagación?
- ¿Cómo deduces esto de las firmas de `fast.log` y los `event_type` de `eve.json`?
- ¿Cuál de los dos representa mayor riesgo inmediato y por qué?

> **[ESCRIBE TU INTERPRETACIÓN AQUÍ]**

---

### 3) Configuración — suricata.yaml y variables de red

**Instrucciones:** Relaciona tu análisis con la configuración de Suricata.

**Preguntas guía:**
- ¿Cómo afectan `HOME_NET` y `EXTERNAL_NET` a las alertas que observaste?
- ¿Hubo alertas que esperabas ver y no aparecieron? ¿A qué puede deberse?
- ¿Qué cambio hiciste en `rule-files`? ¿Por qué es necesario?
- ¿Cuántas reglas cargó `suricata-update`? ¿Sin él, habrías detectado WannaCry?
- ¿Qué ajuste de `HOME_NET` propones para reducir falsos positivos en PCAP 1?

> **[ESCRIBE TU ANÁLISIS DE CONFIGURACIÓN AQUÍ]**

---

### 4) Propuestas de Optimización

**Propuestas concretas basadas en tu análisis:**

| # | Problema observado | Ajuste propuesto | Impacto esperado |
|---|-------------------|-----------------|-----------------|
| 1 | [ej: Demasiadas alertas de baja prioridad en PCAP 1] | [ej: Deshabilitar categoría "Potentially Bad Traffic"] | [ej: -30% alertas, misma cobertura crítica] |
| 2 | [ej: MS17-010 no detectado sin suricata-update] | [ej: Automatizar suricata-update diario con cron] | [ej: Cobertura actualizada contra amenazas recientes] |
| 3 | [COMPLETA] | [COMPLETA] | [COMPLETA] |

In [ ]:
# ── Resumen final de la práctica ─────────────────────────────────────────────
print("=" * 70)
print("📋 RESUMEN FINAL DE LA PRÁCTICA")
print("=" * 70)

def _exists(path):
    try:
        return path.exists()
    except PermissionError:
        result = subprocess.run(['sudo', 'test', '-e', str(path)], capture_output=True)
        return result.returncode == 0

summary = {
    'Suricata instalado':        bool(shutil.which('suricata')),
    'suricata.yaml revisado':    _exists(CONFIG_FILE),
    'Ruta de reglas ajustada':   'suricata.rules' in read_file_safe(CONFIG_FILE),
    'suricata-update ejecutado': _exists(RULES_DIR / 'suricata.rules'),
    'PCAP 1 analizado':          _exists(PCAP1_RESULTS / 'fast.log'),
    'PCAP 2 analizado':          _exists(PCAP2_RESULTS / 'fast.log'),
    'Comparativa generada':      True,
}

rules_count = count_rules(RULES_DIR / 'suricata.rules')

for task, done in summary.items():
    icon = '✅' if done else '❌'
    print(f"  {icon} {task}")

print(f"\n📊 Reglas cargadas: {rules_count['active']:,} activas")

alerts_1 = len(parse_fast_log(PCAP1_RESULTS / 'fast.log')) if _exists(PCAP1_RESULTS / 'fast.log') else 0
alerts_2 = len(parse_fast_log(PCAP2_RESULTS / 'fast.log')) if _exists(PCAP2_RESULTS / 'fast.log') else 0

print(f"\n📄 Alertas generadas:")
print(f"   PCAP 1 (Webserver Scans):      {alerts_1}")
print(f"   PCAP 2 (WannaCry/EternalBlue): {alerts_2}")

print(f"\n📁 Resultados guardados en: {RESULTS_DIR}")
print("\n✅ Práctica completada. Procede a completar el informe técnico (Sección 7).")